# Agent Simulation with evaluatorq

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/orq-ai/orqkit/blob/main/packages/evaluatorq-py/examples/agent_simulation_intro.ipynb)  [![Open in molab](https://marimo.io/shield.svg)](https://molab.marimo.io/github/orq-ai/orqkit/blob/main/packages/evaluatorq-py/examples/agent_simulation_intro.ipynb)

Simulation answers *"does my agent work for real users?"* A **user-simulator LLM** plays a persona pursuing a goal across a multi-turn conversation, and a **judge LLM** scores each result against your criteria. It is the non-adversarial counterpart to [red teaming](red_teaming_intro.ipynb).

This notebook is a 5-minute, **SDK-only** tour. For the CLI (`eq sim`), generated personas/scenarios, datasets, and the production/CI pattern, see [`src/evaluatorq/simulation/README.md`](../src/evaluatorq/simulation/README.md) and the runnable scripts in [`examples/agent_simulation/`](agent_simulation/README.md).

**Prerequisites:** `ORQ_API_KEY` (preferred) or `OPENAI_API_KEY` — drives the user-simulator and judge LLMs. The default `sim_model` is `openai/gpt-5.4-mini` via the ORQ router; if you use OpenAI directly, pass `sim_model="gpt-5.4-mini"` (drop the `openai/` prefix).

In [ ]:
# Install dependencies (skip if evaluatorq is already in your environment).
%pip install "evaluatorq[simulation]" python-dotenv

In [ ]:
# Make sure a key is available before we spend tokens. The user-simulator and judge
# need an LLM key — ORQ_API_KEY (preferred) or OPENAI_API_KEY. Keep it in a .env file
# (or your shell) — never paste it into the notebook.
import os

from dotenv import load_dotenv

load_dotenv()
assert os.getenv("ORQ_API_KEY") or os.getenv("OPENAI_API_KEY"), (
    "No LLM key found. Set ORQ_API_KEY (preferred) or OPENAI_API_KEY in a .env file or your shell."
)
print("Key found ✔")

In [ ]:
from evaluatorq.contracts import Message
from evaluatorq.simulation import simulate
from evaluatorq.simulation.types import (
    CommunicationStyle,
    Criterion,
    EmotionalArc,
    Persona,
    Scenario,
    StartingEmotion,
)

## 1. Define a persona

The persona is *who* the simulated user is — their patience, assertiveness, tone, and how their mood shifts during the chat.

In [ ]:
persona = Persona(
    name="Impatient Customer",
    patience=0.2,
    assertiveness=0.8,
    politeness=0.4,
    technical_level=0.3,
    communication_style=CommunicationStyle.terse,
    background="Received the wrong item and wants a refund urgently",
    emotional_arc=EmotionalArc.escalating,
)

## 2. Define a scenario

The scenario is *what* the user wants and how success is measured. `must_happen` / `must_not_happen` criteria become the judge's checklist.

In [ ]:
scenario = Scenario(
    name="Wrong Item Refund",
    goal="Get a full refund for the wrong item received",
    context="Customer ordered headphones but received a phone case instead",
    starting_emotion=StartingEmotion.frustrated,
    criteria=[
        Criterion(description="Agent asks for order details", type="must_happen"),
        Criterion(description="Agent acknowledges the mistake", type="must_happen"),
        Criterion(description="Agent blames the customer", type="must_not_happen"),
    ],
)

## 3. The agent under test

Any `async` function that takes the conversation and returns the next reply works as a target. Here it's a local mock — no orq.ai *deployment* needed (though `ORQ_API_KEY` is still required: it powers the user-simulator and judge LLMs). Swap in `agent_key=` to test a live orq.ai agent; see step 6.

In [ ]:
async def support_agent(messages: list[Message]) -> str:
    last = (messages[-1].content or "").lower() if messages else ""
    if "refund" in last:
        return "I can help with that. Could you share your order number?"
    if "order" in last or "status" in last:
        return "Let me look that up. What email is on the account?"
    if "thank" in last:
        return "Happy to help! Anything else I can do for you?"
    return "Thanks for reaching out. How can I assist you today?"

## 4. Run the simulation

`simulate()` is async — `await` it at the top level in Jupyter. One persona × one scenario yields one `SimulationResult`.

In [ ]:
# The user-simulator and judge use sim_model (default "openai/gpt-5.4-mini" via the ORQ router).
# Using OpenAI directly? Add sim_model="gpt-5.4-mini" (drop the "openai/" prefix).
results = await simulate(
    evaluation_name="basic-simulation-intro",
    target_callback=support_agent,
    personas=[persona],
    scenarios=[scenario],
    max_turns=6,
    evaluator_names=["goal_achieved", "criteria_met"],
)

assert results, "Simulation produced no results — the run failed; check OTel spans under orq.simulation.pipeline"
result = results[0]

## 5. Inspect the result

Score, termination reason, any broken rules, and the full transcript.

In [ ]:
print(f"Goal achieved:  {result.goal_achieved}")
print(f"Goal score:     {result.goal_completion_score:.2f}")
print(f"Turns:          {result.turn_count}")
print(f"Terminated by:  {result.terminated_by}")
if result.rules_broken:
    print(f"Rules broken:   {result.rules_broken}")

print("\n--- Conversation ---")
for msg in result.messages:
    who = "User " if msg.role == "user" else "Agent"
    print(f"{who}: {msg.content}")

## 6. Simulate an orq.ai agent instead

Drop the mock callback and point `simulate()` at a live deployment with `agent_key=`. Requires `ORQ_API_KEY`.

```python
results = await simulate(
    evaluation_name="support-agent-sim",
    agent_key="my-support-agent",
    personas=[persona],
    scenarios=[scenario],
    max_turns=8,
)
```

Don't have personas yet? `generate_and_simulate(agent_description=...)` invents them for you — see the simulation README.

## Where to go next

- **Concepts, entry points, datasets, CLI:** [`src/evaluatorq/simulation/README.md`](../src/evaluatorq/simulation/README.md)
- **Runnable scripts** (orq deployment, tool-using agents, hardening loop, CI gating): [`examples/agent_simulation/`](agent_simulation/README.md)
- **Red teaming** (the adversarial counterpart — does it break under attack?): [`red_teaming_intro.ipynb`](red_teaming_intro.ipynb)